In [29]:
#This script is to add the gene annotation functions (from a reference annotation file) in the selected network genes
#Nodes and selected edges of interest were listed
anotation <- read.delim("your_path/annotation_file.tsv", sep ='\t', header=TRUE)
nodes <- read.delim("your_path/target_nodes.csv", sep =',', header=TRUE)
edges <- read.delim("your_path/target_edges.csv", sep =',', header=TRUE)

In [30]:
nrow(edges)
colnames(edges)

nrow(nodes)
colnames(nodes)

[1] 329

[1] "interaction"        "name"               "pair"              
[4] "selected"           "shared.interaction" "shared.name"       
[7] "weight"

[1] 330

[1] "DE_genotype_all" "DE_hpi_all"      "mm"              "module"         
[5] "name"            "p_mm"            "selected"        "shared.name"    
[9] "type"

In [ ]:
#Clean data: IDs of genes in your network must be the same IDs of your annotation file
#In this case we selected a central gen "Llu_lncRNA_25730", and we searched their annotated co-expressed genes
 

library(dplyr)
library(stringr)

#gene of interest
gen <- "Llu_lncRNA_25730"

#Just keep interactions between our genes and other nodes
edges <- edges %>%
  mutate(
    name = if_else(
      str_detect(name, paste0("^", gen)), 
      str_replace(name, paste0("^", gen, "\\s*\\(interacts with\\)\\s*"), ""),
      str_replace(name, paste0("\\s*\\(interacts with\\)\\s*", gen, "$"), "")
    )
  )

head(edges)


In [32]:
#Clean columns
edges <- edges %>%
  select(-any_of(c("interaction", "shared interaction", "shared.interaction", "selected")))


nodes <- nodes %>%
  select(-any_of(c("selected", "shared name", "shared.name")))

#Header of genes IDs in your network must be the same to your annotation file
edges <- edges %>%
  rename(Gene.ID = name)
nodes <- nodes %>%
  rename(Gene.ID = name)

In [33]:
nrow(edges)
colnames(edges)

nrow(nodes)
colnames(nodes)

[1] 329

[1] "Gene.ID"     "pair"        "shared.name" "weight"

[1] 330

[1] "DE_genotype_all" "DE_hpi_all"      "mm"              "module"         
[5] "Gene.ID"         "p_mm"            "type"

In [34]:
#merge edges and nodes information
merged <- inner_join(edges, nodes, by = "Gene.ID")


In [35]:
nrow(merged)
colnames(merged)
head(merged)

[1] 329

[1] "Gene.ID"         "pair"            "shared.name"     "weight"         
 [5] "DE_genotype_all" "DE_hpi_all"      "mm"              "module"         
 [9] "p_mm"            "type"

,Gene.ID,pair,shared.name,weight,DE_genotype_all,DE_hpi_all,mm,module,p_mm,type
,<chr>,<chr>,<chr>,<dbl>,<chr>,<chr>,<dbl>,<chr>,<dbl>,<chr>
1,Llu18275,r,Llu18275 (interacts with) Llu_lncRNA_25730,0.3034480,up,NA,0.9494875,MM5,1.203911e-16,codingRNA
2,Llu14903,r,Llu14903 (interacts with) Llu_lncRNA_25730,0.2682810,up,NA,0.9409587,MM5,1.180414e-15,codingRNA
3,Llu36328,r,Llu36328 (interacts with) Llu_lncRNA_25730,0.2398975,up,NA,0.9065887,MM5,9.105393e-13,codingRNA
4,Llu12288,r,Llu12288 (interacts with) Llu_lncRNA_25730,0.5037561,up,NA,0.9912194,MM5,6.353321e-28,codingRNA
5,Llu03994,r84hpi,Llu03994 (interacts with) Llu_lncRNA_25730,0.2587503,up,NA,0.9443998,MM5,4.907953e-16,codingRNA
6,Llu18226,r84hpi,Llu18226 (interacts with) Llu_lncRNA_25730,0.2553158,up,NA,0.8720709,MM11,8.024229e-11,codingRNA


In [36]:
#Now search genes in your annotation file
#Some genes could be non annotated
#Some genes could have more than one annotation 
anotation_final <- inner_join(merged, anotation, by = "Gene.ID")
nrow(anotation_final)
head(anotation_final)

[1] 284

,Gene.ID,pair,shared.name,weight,DE_genotype_all,DE_hpi_all,mm,module,p_mm,type,⋯,KEGG_ko,KEGG_Pathway,KEGG_Module,KEGG_Reaction,KEGG_rclass,BRITE,KEGG_TC,CAZy,BiGG_Reaction,PFAMs
,<chr>,<chr>,<chr>,<dbl>,<chr>,<chr>,<dbl>,<chr>,<dbl>,<chr>,⋯,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
1,Llu18275,r,Llu18275 (interacts with) Llu_lncRNA_25730,0.3034480,up,NA,0.9494875,MM5,1.203911e-16,codingRNA,⋯,,,,,,,,,,
2,Llu14903,r,Llu14903 (interacts with) Llu_lncRNA_25730,0.2682810,up,NA,0.9409587,MM5,1.180414e-15,codingRNA,⋯,,,,,,,,,,
3,Llu36328,r,Llu36328 (interacts with) Llu_lncRNA_25730,0.2398975,up,NA,0.9065887,MM5,9.105393e-13,codingRNA,⋯,,,,,,,,,,
4,Llu12288,r,Llu12288 (interacts with) Llu_lncRNA_25730,0.5037561,up,NA,0.9912194,MM5,6.353321e-28,codingRNA,⋯,,,,,,,,,,
5,Llu03994,r84hpi,Llu03994 (interacts with) Llu_lncRNA_25730,0.2587503,up,NA,0.9443998,MM5,4.907953e-16,codingRNA,⋯,,,,,,,,,,
6,Llu18226,r84hpi,Llu18226 (interacts with) Llu_lncRNA_25730,0.2553158,up,NA,0.8720709,MM11,8.024229e-11,codingRNA,⋯,,,,,,,,,,


In [37]:
write.table(anotation_final, "/mnt/data/lupino/12_WGCNA_2025/5_COEXPRESS_2025/coexp/3_filtro_quantil_bignet/anotados_final/net_pairsxtrait_095_DE_lnc25730_cod_target_anotada.tsv",sep = '\t',row.names = FALSE,quote = FALSE)